# Paper — 02: AP Metrics (Table 1)

**Produces:** `figures_paper/table1_ap.csv` + printed table

Computes COCO-style AP50 and AP50:95 for all pipeline variants using Shapely polygon IoU
on the 5 Prieur et al. ground-truth test tiles.

**No GPU required.** Predictions must already be saved as shapefiles from prior inference runs.

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
from shapely.geometry import box as shapely_box
import warnings

from rastertools_BOULDERING import metadata as raster_metadata

In [ ]:
prieur_dir      = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px")
prieur_test_dir = prieur_dir / "preprocessing" / "test"
work_dir        = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster       = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")

gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

# Per-tile eval: inference run fresh on each GT tile with overlap=0.
# Use *-mask-nms.shp (not *-bbox-nms.shp) for polygon IoU against GT.
tile_eval_dir = work_dir / "exp_tile_eval"

PRED_CONFIGS = {
    "YOLOv8":          (tile_eval_dir, "yolo_*/*-mask-nms.shp"),
    "SAM2 zero-shot":  (tile_eval_dir, "sam2_[0-9]*/*-mask-nms.shp"),
    "SAM2 fine-tuned": (tile_eval_dir, "sam2_ft_*/*-mask-nms.shp"),
}

IOU_THRESHOLDS = np.arange(0.50, 1.00, 0.05)   # 0.50, 0.55, …, 0.95

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
print(f"Resolution: {res:.4f} m/px   Areal threshold: {AREAL_THRESHOLD:.4f} m²")

In [ ]:
def poly_iou(p1, p2):
    """Shapely IoU between two polygons (area-based)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        p1b, p2b = p1.buffer(0), p2.buffer(0)
        inter = p1b.intersection(p2b).area
        uni   = p1b.union(p2b).area
    return inter / uni if uni > 0 else 0.0


def compute_ap(gt_polys, pred_polys, pred_scores, iou_threshold):
    """Greedy one-to-one matching; 101-point interpolated PR curve."""
    if not pred_polys or not gt_polys:
        return 0.0

    order      = np.argsort(pred_scores)[::-1]
    pred_polys = [pred_polys[i] for i in order]

    matched_gt = set()
    tp = np.zeros(len(pred_polys))
    fp = np.zeros(len(pred_polys))

    for i, pred in enumerate(pred_polys):
        best_iou, best_j = 0.0, -1
        for j, gt in enumerate(gt_polys):
            if j in matched_gt:
                continue
            iou = poly_iou(pred, gt)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_threshold:
            tp[i] = 1
            matched_gt.add(best_j)
        else:
            fp[i] = 1

    tp_cum    = np.cumsum(tp)
    fp_cum    = np.cumsum(fp)
    recall    = tp_cum / len(gt_polys)
    precision = tp_cum / (tp_cum + fp_cum + 1e-9)

    ap = sum(
        (precision[recall >= t].max() if (recall >= t).any() else 0.0)
        for t in np.linspace(0, 1, 101)
    ) / 101
    return ap

In [ ]:
# Load GT per tile — keep spatial index for fast pred filtering
gt_per_tile  = {}
all_gt_polys = []

for tile_id in gt_tile_ids:
    shp = prieur_test_dir / "labels" / f"M1221383405_{tile_id}_mask.shp"
    if not shp.exists():
        print(f"  Tile {tile_id}: GT shapefile not found — skipping")
        continue
    gdf = gpd.read_file(shp)
    gdf["poly_area"] = gdf.geometry.area
    gdf = gdf[gdf["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
    gt_per_tile[tile_id] = gdf
    all_gt_polys.extend(list(gdf.geometry))
    print(f"  Tile {tile_id}: {len(gdf)} GT boulders")

print(f"\nTotal GT boulders across {len(gt_per_tile)} tiles: {len(all_gt_polys)}")

# Build the spatial footprint of all GT tiles for pred filtering
gt_footprint = None
for gdf in gt_per_tile.values():
    bounds   = gdf.total_bounds
    tile_box = shapely_box(*bounds)
    gt_footprint = tile_box if gt_footprint is None else gt_footprint.union(tile_box)

In [ ]:
ap_results = {}

for name, (pred_dir, glob) in PRED_CONFIGS.items():
    shp_paths = sorted(pred_dir.glob(glob))
    if not shp_paths:
        print(f"\n[{name}] No shapefiles found at {pred_dir}/{glob} — skipping")
        continue

    print(f"\n[{name}] Found {len(shp_paths)} shapefile(s)...")
    gdfs = [gpd.read_file(p) for p in shp_paths]
    gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    # No area filter on predictions — SAM2 tight masks would be unfairly penalised.
    # Area filter applies to GT only (load-gt cell).

    score_col = "confidence" if "confidence" in gdf.columns else \
                "score"      if "score"      in gdf.columns else None
    gdf["_score"] = gdf[score_col].astype(float) if score_col else 1.0

    # Restrict to GT tile footprint
    mask     = gdf.geometry.intersects(gt_footprint.buffer(5))
    gdf_eval = gdf[mask].copy()
    print(f"  {len(gdf_eval)} predictions within GT tile footprint")

    pred_polys  = list(gdf_eval.geometry)
    pred_scores = gdf_eval["_score"].values

    aps = []
    for thresh in tqdm(IOU_THRESHOLDS, desc="  thresholds", leave=False):
        aps.append(compute_ap(all_gt_polys, pred_polys, pred_scores, thresh))

    ap_results[name] = {
        "AP50":    round(aps[0], 4),
        "AP50:95": round(float(np.mean(aps)), 4),
    }
    print(f"  AP50={aps[0]:.3f}   AP50:95={np.mean(aps):.3f}")

In [ ]:
print("\nTable 1: Segmentation AP")
print(f"{'Method':<24}  {'AP50':>6}  {'AP50:95':>8}")
print("-" * 44)
for name, r in ap_results.items():
    print(f"{name:<24}  {r['AP50']:>6.3f}  {r['AP50:95']:>8.3f}")

df_table = pd.DataFrame(ap_results).T[["AP50", "AP50:95"]]
df_table.to_csv(OUT_DIR / "table1_ap.csv")
print("\nSaved table1_ap.csv")